# Phát hiện chuyển động đám đông bất thường trên UMN

**Mục tiêu.** Học mẫu chuyển động *bình thường* rồi phát hiện hoảng loạn hoặc chạy tán loạn. Dữ liệu train chỉ gồm đoạn bình thường, không cần nhãn bất thường.

**Đặc trưng.** Mỗi khung chia thành lưới 8×8 ô. Mỗi ô có:
- optical flow (Farneback) **chỉ lấy trong vùng có người**, theo mask của MobileCount + Mask;
- mật độ người từ MobileCount + Mask.

**Mô hình.** Autoencoder tích chập trên cửa sổ 16 khung. Điểm bất thường là sai số tái tạo; ô có sai số lớn cho biết bất thường xảy ra ở đâu.

**So sánh.**
- B0: luật "hỗn loạn" của demo (tốc độ và độ hỗn loạn hướng).
- B1: Isolation Forest trên đặc trưng toàn khung.
- 3 biến thể autoencoder (ablation đóng góp của mô hình đếm):

| Biến thể | Đặc trưng |
|---|---|
| AE-flow | Flow toàn ảnh, không dùng mô hình đếm |
| AE-flow·mask | Flow chỉ trong mask người |
| AE-flow·mask+density | Như trên, thêm kênh mật độ và mask |

**Giao thức.** Theo đúng Marsden và cộng sự, *Holistic Features for Real-Time Crowd Behaviour Anomaly Detection*, ICIP 2016 (arXiv:1606.05310):
- Dataset UMN: 11 clip, 3 bối cảnh, 7.739 khung, 320×240, 30 fps.
- Với mỗi bối cảnh, **200 khung đầu của mỗi clip** gộp lại làm tập train (toàn bộ là khung bình thường). **Mọi khung còn lại** là tập test.
- Mỗi bối cảnh có một mô hình riêng.
- Chỉ số: AUC theo khung. Bài báo này báo cáo AUC của MDT, CM, SFM và của chính họ dưới cùng giao thức, nên so sánh được trực tiếp.
- 20 khung cuối trong 200 khung train của mỗi clip được dùng làm validation để chọn epoch. Các khung này vẫn nằm trong phần train của giao thức, nên tập test không thay đổi.

**Cài đặt Kaggle.**
- Bật **Internet**: video được tải từ GitHub.
- Add Input: Model `chinhde/mobilecountmask` (version 1, có `mc_mask_f4_partB.pth` = MobileCount + Mask, fold 4).
- GPU T4.

## 1. Cấu hình

In [ ]:
import copy, json, math, random, time, urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve

SEED = 42
UMN_URL = ('https://raw.githubusercontent.com/vishal-siddegowda/CrowdAnomalyDetection_DeepLearning/'
           'master/UMN/Final%20UMN%20dataset/UMN%20video%20data.avi')
WORK = Path('/kaggle/working')
UMN_AVI = WORK / 'umn.avi'
WEIGHT_PATH = Path('/kaggle/input/models/chinhde/mobilecountmask/pytorch/default/1/mc_mask_f4_partB.pth')

TEXT_BOX = (0, 0, 176, 33)      # x0, y0, x1, y1: vùng chữ "Abnormal Crowd Activity" bị che ở mọi khung
RED_MIN_PIXELS = 300            # số điểm ảnh đỏ trong 30 dòng đầu để coi khung là bất thường
SCENE_CUT_DIFF = 30.0           # chênh lệch xám trung bình giữa 2 khung liền nhau khi đổi bối cảnh

INFER_WIDTH = 640               # phóng UMN 320×240 lên 640×480 cho MobileCount (người UMN rất nhỏ)
GRID = (8, 8)                   # số ô theo hàng, cột
NBINS = 8                       # số hướng trong histogram flow
WIN = 16                        # số khung mỗi cửa sổ thời gian
TRAIN_FRAMES = 200              # giao thức Marsden et al. (ICIP 2016): 200 khung đầu mỗi clip để train
VAL_FRAMES = 20                 # 20 khung cuối trong phần train đó dùng làm validation chọn epoch
SMOOTH = 15                     # trung bình trượt nhân quả của điểm bất thường (0,5 s)
TOPK = 3                        # điểm khung = trung bình 3 ô sai số lớn nhất

AE_EPOCHS, AE_LR, AE_BATCH = 80, 1e-3, 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


seed_all()
WORK.mkdir(parents=True, exist_ok=True)
print('device:', DEVICE)

## 2. Tải video UMN và đọc toàn bộ khung

In [ ]:
if not UMN_AVI.exists():
    urllib.request.urlretrieve(UMN_URL, UMN_AVI)
print('kích thước file:', UMN_AVI.stat().st_size, 'byte')

cap = cv2.VideoCapture(str(UMN_AVI))
FPS = cap.get(cv2.CAP_PROP_FPS)
buf = []
while True:
    ok, f = cap.read()
    if not ok:
        break
    buf.append(f)
cap.release()
frames = np.stack(buf); del buf
N, H, W = frames.shape[:3]
print(f'{N} khung, {W}×{H}, {FPS:.0f} fps')
assert (W, H) == (320, 240) and N > 7700

## 3. Nhãn, clip và bối cảnh

- **Nhãn.** Video gốc chỉ hiện chữ đỏ "Abnormal Crowd Activity" ở góc trên trái trong lúc bất thường, nên nhãn của từng khung đọc từ chữ này.
- **Clip.** Mỗi clip kết thúc ngay sau đoạn bất thường của nó. Nếu trong 30 khung sau đó có đổi bối cảnh thì clip kéo dài tới chỗ đổi.
- **Che chữ.** Sau khi đọc nhãn, vùng chữ bị **che ở mọi khung**. Nếu không che, đặc trưng sẽ lộ nhãn.

In [ ]:
top = frames[:, :30].astype(np.int16)
red_px = ((top[..., 2] > 150) & (top[..., 1] < 90) & (top[..., 0] < 90)).sum(axis=(1, 2))
label = (red_px > RED_MIN_PIXELS).astype(np.int8)
del top

gray = np.stack([cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames])
fdiff = np.r_[0.0, np.abs(np.diff(gray.astype(np.int16), axis=0)).mean(axis=(1, 2))]
scene_cuts = np.flatnonzero(fdiff > SCENE_CUT_DIFF)
print('đổi bối cảnh tại khung:', scene_cuts.tolist())

edges = np.flatnonzero(np.diff(np.r_[0, label, 0]))
abn = list(zip(edges[::2], edges[1::2] - 1))
assert len(abn) == 11, f'cần 11 đoạn bất thường, tìm được {len(abn)}'

clips, start = [], 0
for k, (a, b) in enumerate(abn):
    if k == len(abn) - 1:
        end = N - 1
    else:
        near = scene_cuts[(scene_cuts > b) & (scene_cuts <= b + 30)]
        end = near[0] - 1 if len(near) else b
    clips.append(dict(clip=k + 1, start=start, end=end, abn_start=a, abn_end=b))
    start = end + 1

clip_id = np.zeros(N, int)
scene_id = np.searchsorted(scene_cuts, np.arange(N), side='right')
for c in clips:
    clip_id[c['start']:c['end'] + 1] = c['clip']
clips = pd.DataFrame(clips)
clips['scene'] = scene_id[clips.start]
assert clips.groupby('scene').size().tolist() == [2, 6, 3]
print(f'tỉ lệ khung bất thường: {label.mean():.3f}')
display(clips)

x0, y0, x1, y1 = TEXT_BOX
frames[:, y0:y1, x0:x1] = 128
gray[:, y0:y1, x0:x1] = 128

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
picks = [clips.start[0] + 100, clips.abn_start[0] + 30, clips.start[3] + 100, clips.abn_start[3] + 20,
         clips.start[5] + 100, clips.abn_start[5] + 20, clips.start[8] + 100, clips.abn_start[8] + 15,
         clips.start[9] + 100, clips.abn_start[9] + 10, clips.start[10] + 100, clips.abn_start[10] + 8]
for ax, t in zip(axes.ravel(), picks):
    ax.imshow(cv2.cvtColor(frames[t], cv2.COLOR_BGR2RGB))
    ax.set_title(f'khung {t} · clip {clip_id[t]} · {"BẤT THƯỜNG" if label[t] else "bình thường"}')
    ax.axis('off')
plt.tight_layout(); plt.savefig(WORK / 'umn_samples.png', dpi=110); plt.show()

plt.figure(figsize=(16, 2))
plt.fill_between(np.arange(N), label, step='mid', color='tab:red', alpha=.5, label='bất thường')
for s in clips.start[1:]:
    plt.axvline(s, color='k', lw=.6)
plt.yticks([]); plt.xlabel('khung'); plt.title('Nhãn theo thời gian, vạch đen = ranh giới clip'); plt.legend()
plt.tight_layout(); plt.savefig(WORK / 'umn_labels.png', dpi=110); plt.show()

## 4. Chia train / test theo Marsden và cộng sự (ICIP 2016)

- Với mỗi bối cảnh, **200 khung đầu của mỗi clip** gộp lại làm tập train. Trong 200 khung đó, 20 khung cuối làm validation để chọn epoch.
- **Mọi khung còn lại** dùng để test.
- Tập test chỉ dùng một lần khi đánh giá.

In [ ]:
assert (clips.abn_start - clips.start > TRAIN_FRAMES).all(), 'có clip bắt đầu bất thường trước khung 200'
split = np.full(N, 'test', dtype=object)
for c in clips.itertuples():
    split[c.start:c.start + TRAIN_FRAMES - VAL_FRAMES] = 'train'
    split[c.start + TRAIN_FRAMES - VAL_FRAMES:c.start + TRAIN_FRAMES] = 'val'
assert not ((split != 'test') & (label == 1)).any()
print(pd.crosstab(scene_id, split).rename_axis(index='bối cảnh'))
print('khung test bất thường:', int(((split == 'test') & (label == 1)).sum()))

## 5. Mô hình đếm: MobileCount + Mask

Load đúng file `mc_mask_f4_partB.pth` (fold 4 của `final_mobilecount_cv`). Kiến trúc giống notebook train và demo.

Mô hình cho ra hai bản đồ ở 1/4 độ phân giải: bản đồ mật độ (đã nhân với mask) và xác suất có người (mask).

In [ ]:
DENSITY_SCALE = 100.0
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


class ChainedResidualPool(nn.Module):
    def __init__(self, channels, n_stages=4):
        super().__init__()
        self.pool = nn.MaxPool2d(5, stride=1, padding=2)
        self.convs = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(n_stages)])

    def forward(self, x):
        top = x
        for conv in self.convs:
            top = conv(self.pool(top))
            x = x + top
        return x


class MobileCountMask(nn.Module):
    # Kiến trúc của final_mobilecount_cv / demo: MobileNetV2 + Light-Weight RefineNet + nhánh mask.
    def __init__(self, channels=64):
        super().__init__()
        f = torchvision.models.mobilenet_v2(weights=None).features
        self.encoder = nn.ModuleList([f[:4], f[4:7], f[7:14], f[14:18]])
        self.project = nn.ModuleList([nn.Conv2d(c, channels, 1, bias=False) for c in (24, 32, 96)])
        self.deep = nn.Sequential(nn.Conv2d(320, channels, 1, bias=False), nn.ReLU(inplace=True))
        self.crp = nn.ModuleList([ChainedResidualPool(channels) for _ in range(4)])
        self.fuse = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(3)])
        self.head = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(channels, 1, 1))
        self.mask_head = nn.Sequential(nn.Conv2d(channels, 32, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1))

    def forward(self, x):
        feats = []
        for stage in self.encoder:
            x = stage(x); feats.append(x)
        y = self.crp[3](self.deep(feats[3]))
        for i in (2, 1, 0):
            y = F.interpolate(self.fuse[i](y), size=feats[i].shape[-2:], mode='bilinear', align_corners=False)
            y = self.crp[i](F.relu(self.project[i](feats[i]) + y))
        m = torch.sigmoid(self.mask_head(y))
        return self.head(y) * m, m


def load_counter(path):
    if not path.exists():
        raise FileNotFoundError(f'Không thấy {path}: hãy Add Input model chinhde/mobilecountmask (version 1).')
    state = torch.load(path, map_location='cpu', weights_only=True)
    assert 'deep.0.weight' in state and any(k.startswith('mask_head') for k in state),         'checkpoint không phải MobileCount + Mask (thiếu deep.* hoặc mask_head.*)'
    model = MobileCountMask()
    model.load_state_dict(state)
    print('MobileCount + Mask:', path, f'| {sum(p.numel() for p in model.parameters()) / 1e6:.2f} triệu tham số')
    return model.eval().to(DEVICE)


counter = load_counter(WEIGHT_PATH)

## 6. Trích đặc trưng

Mỗi khung được chạy qua MobileCount để lấy mật độ và mask, rồi tính optical flow Farneback so với khung trước.

| Nhóm | Hình dạng | Nội dung |
|---|---|---|
| `F0` | 8×8×11 | Flow toàn ảnh: tốc độ, histogram 8 hướng, entropy hướng, độ đồng hướng |
| `FM` | 8×8×11 | Như `F0` nhưng có trọng số theo mask người |
| `D` | 8×8×2 | Tổng mật độ (người) và tỉ lệ mask của ô |
| `G` | 5 | Đặc trưng toàn khung (có mask): tốc độ, entropy, độ đồng hướng, tỉ lệ ngược chiều, tổng số người |

Khung đầu của mỗi clip có flow = 0, vì khung trước nó thuộc clip khác. Kết quả được cache vào `umn_feats.npz`.

In [ ]:
GH, GW = GRID
CH, CW = H // GH, W // GW
IW, IH = INFER_WIDTH, round(INFER_WIDTH * H / W)


def to_cells(a):
    # (H, W, ...) -> (GH, GW, CH, CW, ...)
    return a.reshape(GH, CH, GW, CW, *a.shape[2:]).swapaxes(1, 2)


@torch.no_grad()
def run_counter(batch_bgr):
    x = np.stack([cv2.cvtColor(cv2.resize(f, (IW, IH), interpolation=cv2.INTER_LINEAR), cv2.COLOR_BGR2RGB)
                  for f in batch_bgr]).astype(np.float32) / 255.0
    x = torch.from_numpy(((x - MEAN) / STD).transpose(0, 3, 1, 2)).to(DEVICE)
    den, mprob = counter(x)
    return den[:, 0].float().cpu().numpy() / DENSITY_SCALE, mprob[:, 0].float().cpu().numpy()


def flow_cell_feats(flow, w):
    mag = np.hypot(flow[..., 0], flow[..., 1])
    ang = np.arctan2(flow[..., 1], flow[..., 0])
    b = ((ang + np.pi) / (2 * np.pi) * NBINS).astype(np.int64) % NBINS
    wm = w * mag
    cw, cwm = to_cells(w).sum((2, 3)), to_cells(wm).sum((2, 3))
    speed = cwm / (cw + 1e-6)
    bc, wmc = to_cells(b), to_cells(wm)
    hist = np.stack([(wmc * (bc == k)).sum((2, 3)) for k in range(NBINS)], -1)
    p = hist / (cwm[..., None] + 1e-6)
    ent = -(p * np.log(p + 1e-9)).sum(-1) / np.log(NBINS)
    vx, vy = to_cells(w * flow[..., 0]).sum((2, 3)), to_cells(w * flow[..., 1]).sum((2, 3))
    coh = np.hypot(vx, vy) / (cwm + 1e-6)
    return np.concatenate([speed[..., None], p, ent[..., None], coh[..., None]], -1).astype(np.float32)


def global_feats(flow, w, count):
    mag = np.hypot(flow[..., 0], flow[..., 1])
    ang = np.arctan2(flow[..., 1], flow[..., 0])
    wm = w * mag
    tot = wm.sum() + 1e-6
    speed = wm.sum() / (w.sum() + 1e-6)
    b = ((ang + np.pi) / (2 * np.pi) * NBINS).astype(np.int64) % NBINS
    p = np.bincount(b.ravel(), weights=wm.ravel(), minlength=NBINS) / tot
    ent = -(p * np.log(p + 1e-9)).sum() / np.log(NBINS)
    vx, vy = (w * flow[..., 0]).sum(), (w * flow[..., 1]).sum()
    coh = np.hypot(vx, vy) / tot
    dev = np.abs(np.angle(np.exp(1j * (ang - np.arctan2(vy, vx)))))
    counter_ratio = (wm * (dev > 2 * np.pi / 3)).sum() / tot
    return np.array([speed, ent, coh, counter_ratio, count], np.float32)


FEAT_CACHE = WORK / 'umn_feats.npz'
if FEAT_CACHE.exists():
    z = np.load(FEAT_CACHE); F0, FM, D, G = z['F0'], z['FM'], z['D'], z['G']
    print('đọc cache', FEAT_CACHE)
else:
    t0 = time.time()
    F0 = np.zeros((N, GH, GW, NBINS + 3), np.float32)
    FM = np.zeros_like(F0)
    D = np.zeros((N, GH, GW, 2), np.float32)
    G = np.zeros((N, 5), np.float32)
    ones = np.ones((H, W), np.float32)
    clip_start = set(clips.start.tolist())
    for s in range(0, N, 64):
        den, mprob = run_counter(frames[s:s + 64])
        for j in range(len(den)):
            t = s + j
            dh, dw = den[j].shape
            D[t, ..., 0] = den[j].reshape(GH, dh // GH, GW, dw // GW).sum((1, 3))
            mpx = cv2.resize(mprob[j], (W, H), interpolation=cv2.INTER_LINEAR)
            D[t, ..., 1] = to_cells(mpx).mean((2, 3))
            if t in clip_start:
                flow = np.zeros((H, W, 2), np.float32)
            else:
                flow = cv2.calcOpticalFlowFarneback(gray[t - 1], gray[t], None, 0.5, 3, 15, 3, 5, 1.2, 0)
            F0[t] = flow_cell_feats(flow, ones)
            FM[t] = flow_cell_feats(flow, mpx)
            G[t] = global_feats(flow, mpx, den[j].sum())
        if s % 1024 == 0:
            print(f'  {s}/{N}  {time.time() - t0:.0f}s')
    np.savez_compressed(FEAT_CACHE, F0=F0, FM=FM, D=D, G=G)
    print(f'trích đặc trưng xong trong {time.time() - t0:.0f}s')

print('số người trung bình/khung theo bối cảnh:', [round(float(G[scene_id == k, 4].mean()), 1) for k in range(3)])

## 7. Cửa sổ thời gian và chuẩn hóa

- Điểm của khung *t* dựa trên cửa sổ `[t-15 … t]` trong cùng clip, tức là **nhân quả** và chạy được thời gian thực. Ở đầu clip, khung đầu tiên được lặp lại để đủ 16 khung.
- Chuẩn hóa z-score riêng cho từng (bối cảnh, ô, kênh), với thống kê lấy **chỉ từ khung train**.
- Vì vậy mô hình học mức "bình thường" riêng cho từng ô: người ở xa camera đi chậm hơn trên ảnh không bị coi là bất thường. Cách này thích nghi với góc camera mà không cần hiệu chỉnh phối cảnh.

In [ ]:
clip_first = clips.set_index('clip').start.reindex(clip_id).to_numpy()
WIDX = np.maximum(np.arange(N)[:, None] - np.arange(WIN)[::-1][None, :], clip_first[:, None])   # (N, WIN)

VARIANTS = {
    'AE-flow': lambda: F0,
    'AE-flow·mask': lambda: FM,
    'AE-flow·mask+density': lambda: np.concatenate([FM, D], -1),
}


def normalize_scene(X, scene):
    # X: (N, GH, GW, C). Trả về tensor (N_scene, C, GH, GW) đã chuẩn hóa và chỉ số khung của bối cảnh.
    idx = np.flatnonzero(scene_id == scene)
    tr = idx[split[idx] == 'train']
    mu, sd = X[tr].mean(0), X[tr].std(0)
    Xn = np.clip((X[idx] - mu) / (sd + 1e-3), -10, 10).transpose(0, 3, 1, 2)
    return torch.from_numpy(np.ascontiguousarray(Xn)).float().to(DEVICE), idx, (mu, sd)


def causal_smooth(score):
    out = np.empty_like(score)
    for c in clips.itertuples():
        s = score[c.start:c.end + 1]
        cs = np.cumsum(np.r_[0.0, s])
        k = np.arange(1, len(s) + 1)
        lo = np.maximum(0, k - SMOOTH)
        out[c.start:c.end + 1] = (cs[k] - cs[lo]) / (k - lo)
    return out

## 8. Mô hình

| | Mô hình | Huấn luyện |
|---|---|---|
| **B0** | Luật "hỗn loạn" của demo, ở dạng liên tục: `min(z(tốc độ), z(entropy hướng))`. z dùng trung vị và MAD của khung train | Không cần train |
| **B1** | Isolation Forest trên 5 đặc trưng toàn khung `G` | Chỉ khung train |
| **AE** | Autoencoder tích chập: 16 khung × C kênh được gộp thành kênh, trên lưới 8×8; nút cổ chai 16 kênh mỗi ô | Chỉ khung train; chọn epoch có loss validation thấp nhất |

In [ ]:
class GridAE(nn.Module):
    def __init__(self, cin, hid=64, code=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(cin, hid, 3, padding=1), nn.ReLU(inplace=True),
                                 nn.Conv2d(hid, code, 3, padding=1), nn.ReLU(inplace=True))
        self.dec = nn.Sequential(nn.Conv2d(code, hid, 3, padding=1), nn.ReLU(inplace=True),
                                 nn.Conv2d(hid, cin, 3, padding=1))

    def forward(self, x):
        return self.dec(self.enc(x))


def windows(Xn, pos_of, frames_idx):
    # Xn: (N_scene, C, GH, GW). Trả về (B, WIN*C, GH, GW)
    w = torch.from_numpy(pos_of[WIDX[frames_idx]]).to(DEVICE)
    x = Xn[w]
    return x.reshape(x.shape[0], -1, GH, GW)


def train_ae(Xn, idx, label_name):
    seed_all()
    pos_of = np.full(N, -1); pos_of[idx] = np.arange(len(idx))
    tr, va = idx[split[idx] == 'train'], idx[split[idx] == 'val']
    model = GridAE(WIN * Xn.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=AE_LR)
    best, best_state, hist = math.inf, None, []
    for ep in range(AE_EPOCHS):
        model.train()
        perm = np.random.permutation(tr)
        tl = 0.0
        for i in range(0, len(perm), AE_BATCH):
            xb = windows(Xn, pos_of, perm[i:i + AE_BATCH])
            loss = F.mse_loss(model(xb), xb)
            opt.zero_grad(); loss.backward(); opt.step()
            tl += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            vl = sum(F.mse_loss(model(xb := windows(Xn, pos_of, va[i:i + 256])), xb, reduction='sum').item()
                     for i in range(0, len(va), 256)) / (len(va) * WIN * Xn.shape[1] * GH * GW)
        hist.append((tl / len(tr), vl))
        if vl < best:
            best, best_state = vl, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    print(f'  {label_name}: val loss tốt nhất {best:.4f} (epoch {int(np.argmin([h[1] for h in hist])) + 1})')
    return model, pos_of, hist


@torch.no_grad()
def ae_score(model, Xn, pos_of, idx):
    model.eval()
    scores, maps = np.zeros(len(idx), np.float32), np.zeros((len(idx), GH, GW), np.float32)
    for i in range(0, len(idx), 256):
        xb = windows(Xn, pos_of, idx[i:i + 256])
        err = (model(xb) - xb).pow(2).reshape(len(xb), WIN, -1, GH, GW).mean((1, 2))
        maps[i:i + len(xb)] = err.cpu().numpy()
        scores[i:i + len(xb)] = err.flatten(1).topk(TOPK, dim=1).values.mean(1).cpu().numpy()
    return scores, maps


def robust_z(x, ref):
    med = np.median(ref)
    return (x - med) / (1.4826 * np.median(np.abs(ref - med)) + 1e-6)

## 9. Huấn luyện và chấm điểm từng bối cảnh

In [ ]:
SCORES = {m: np.full(N, np.nan, np.float32) for m in ['B0 luật demo', 'B1 IsolationForest', *VARIANTS]}
ERRMAPS, AE_MODELS, AE_HIST = {}, {}, {}

for scene in range(3):
    idx = np.flatnonzero(scene_id == scene)
    ref = idx[split[idx] != 'test']
    print(f'— Bối cảnh {scene + 1}: {len(idx)} khung, train {int((split[idx] == "train").sum())}, '
          f'val {int((split[idx] == "val").sum())}, test {int((split[idx] == "test").sum())}')

    SCORES['B0 luật demo'][idx] = np.minimum(robust_z(G[idx, 0], G[ref, 0]), robust_z(G[idx, 1], G[ref, 1]))

    gm, gs = G[ref].mean(0), G[ref].std(0) + 1e-6
    iso = IsolationForest(n_estimators=300, random_state=SEED).fit((G[idx[split[idx] == 'train']] - gm) / gs)
    SCORES['B1 IsolationForest'][idx] = -iso.score_samples((G[idx] - gm) / gs)

    for name, get in VARIANTS.items():
        Xn, sidx, norm = normalize_scene(get(), scene)
        model, pos_of, hist = train_ae(Xn, sidx, name)
        s, maps = ae_score(model, Xn, pos_of, sidx)
        SCORES[name][sidx] = s
        ERRMAPS[(name, scene)] = maps
        AE_MODELS[(name, scene)] = (model, norm)
        AE_HIST[(name, scene)] = hist

SMOOTHED = {m: causal_smooth(np.nan_to_num(s)) for m, s in SCORES.items()}

## 10. Đánh giá trên tập test

- AUC và EER theo khung, tính riêng cho từng bối cảnh.
- **Trung bình**: trung bình AUC của 3 bối cảnh.
- **Gộp**: trước khi gộp các bối cảnh, điểm của mỗi bối cảnh được chuẩn hóa bằng trung vị và MAD của khung validation, vì mỗi bối cảnh có một mô hình riêng.

In [ ]:
def eer(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    i = np.nanargmin(np.abs((1 - tpr) - fpr))
    return (fpr[i] + 1 - tpr[i]) / 2


test = split == 'test'
rows, POOLED = [], {}
for m, s in SMOOTHED.items():
    row = {'Phương pháp': m}
    pooled = np.zeros(N, np.float32)
    for scene in range(3):
        sel = test & (scene_id == scene)
        row[f'AUC bối cảnh {scene + 1}'] = roc_auc_score(label[sel], s[sel])
        val = (split == 'val') & (scene_id == scene)
        pooled[scene_id == scene] = robust_z(s[scene_id == scene], s[val])
    row['AUC trung bình'] = np.mean([row[f'AUC bối cảnh {k + 1}'] for k in range(3)])
    row['AUC gộp'] = roc_auc_score(label[test], pooled[test])
    row['EER gộp'] = eer(label[test], pooled[test])
    POOLED[m] = pooled
    rows.append(row)
results = pd.DataFrame(rows).set_index('Phương pháp')
display(results.style.format('{:.3f}').highlight_max(axis=0, subset=[c for c in results.columns if 'AUC' in c]))
results.to_csv(WORK / 'umn_results.csv')

In [ ]:
# Bảng 3 của Marsden et al. (ICIP 2016, arXiv:1606.05310). Dòng 'cùng giao thức' là kết quả của chính
# bài đó với cách chia 200 khung đầu mỗi clip để train (notebook này dùng đúng cách chia ấy).
PAPERS = pd.DataFrame([
    ('Marsden et al., ICIP 2016 – đặc trưng holistic + GMM', 0.929, 'cùng giao thức'),
    ('Social Force Model (Mehran et al., CVPR 2009)', 0.97, 'Marsden trích lại'),
    ('Commotion Measure (Mousavi et al., ICIP 2015)', 0.98, 'Marsden trích lại'),
    ('MDT (Mahadevan et al., CVPR 2010)', 0.995, 'Marsden trích lại'),
], columns=['Phương pháp', 'AUC', 'Nguồn số liệu'])
ours = results[['AUC trung bình', 'AUC gộp']].reset_index()
ours = pd.concat([
    ours.assign(**{'Phương pháp': 'Đề tài · ' + ours['Phương pháp'] + ' (TB 3 bối cảnh)', 'AUC': ours['AUC trung bình']}),
    ours.assign(**{'Phương pháp': 'Đề tài · ' + ours['Phương pháp'] + ' (gộp)', 'AUC': ours['AUC gộp']}),
])[['Phương pháp', 'AUC']].assign(**{'Nguồn số liệu': 'notebook này'})
cmp = pd.concat([PAPERS, ours]).sort_values('AUC').reset_index(drop=True)
display(cmp.style.format({'AUC': '{:.3f}'}))
cmp.to_csv(WORK / 'umn_vs_papers.csv', index=False)
print('Đối chiếu chính: Marsden et al. (0,929) dùng đúng cách chia train/test như notebook này. '
      'Các dòng "Marsden trích lại" lấy từ bài gốc, có thể dùng cách chia khác, nên chỉ để tham khảo.')

## 11. Hình minh họa

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for m, p in POOLED.items():
    fpr, tpr, _ = roc_curve(label[test], p[test])
    ax.plot(fpr, tpr, label=f'{m} (AUC {results.loc[m, "AUC gộp"]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=.8)
ax.set_xlabel('Tỉ lệ báo nhầm (FPR)'); ax.set_ylabel('Tỉ lệ phát hiện (TPR)'); ax.set_title('ROC trên UMN (gộp 3 bối cảnh)')
ax.legend(fontsize=8, loc='lower right'); plt.tight_layout(); plt.savefig(WORK / 'umn_roc.png', dpi=130); plt.show()

In [ ]:
show = ['B0 luật demo', 'AE-flow', 'AE-flow·mask+density']
fig, axes = plt.subplots(3, 1, figsize=(16, 9))
for scene, ax in enumerate(axes):
    idx = np.flatnonzero(scene_id == scene)
    ax.fill_between(idx, 0, 1, where=label[idx] == 1, color='tab:red', alpha=.2, transform=ax.get_xaxis_transform(), label='bất thường (nhãn)')
    ax.fill_between(idx, 0, 1, where=split[idx] != 'test', color='0.85', alpha=.6, transform=ax.get_xaxis_transform(), label='train/val')
    for m in show:
        p = POOLED[m][idx]
        ax.plot(idx, np.clip(p, -5, 30), lw=1, label=m)
    for s in clips.start[clips.scene == scene]:
        ax.axvline(s, color='k', lw=.5)
    ax.set_ylabel(f'Bối cảnh {scene + 1}')
axes[0].legend(ncol=5, fontsize=8, loc='upper left'); axes[-1].set_xlabel('khung')
fig.suptitle('Điểm bất thường theo thời gian (đã chuẩn hóa theo validation)')
plt.tight_layout(); plt.savefig(WORK / 'umn_timeline.png', dpi=120); plt.show()

In [ ]:
name = 'AE-flow·mask+density'
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for scene in range(3):
    idx = np.flatnonzero(scene_id == scene)
    maps = ERRMAPS[(name, scene)]
    sel = np.flatnonzero((label[idx] == 1) & (split[idx] == 'test'))
    j = sel[np.argmax(SCORES[name][idx][sel])]
    t = idx[j]
    heat = cv2.resize(maps[j], (W, H), interpolation=cv2.INTER_NEAREST)
    rgb = cv2.cvtColor(frames[t], cv2.COLOR_BGR2RGB)
    axes[0, scene].imshow(rgb); axes[0, scene].set_title(f'Bối cảnh {scene + 1} · khung {t} (bất thường)'); axes[0, scene].axis('off')
    axes[1, scene].imshow(rgb); axes[1, scene].imshow(heat, cmap='jet', alpha=.45)
    axes[1, scene].set_title('Sai số tái tạo theo ô (vị trí bất thường)'); axes[1, scene].axis('off')
plt.tight_layout(); plt.savefig(WORK / 'umn_localization.png', dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for scene, ax in enumerate(axes):
    for name in VARIANTS:
        h = np.array(AE_HIST[(name, scene)])
        ax.plot(h[:, 1], label=name)
    ax.set_title(f'Loss validation · bối cảnh {scene + 1}'); ax.set_xlabel('epoch'); ax.set_yscale('log')
axes[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(WORK / 'umn_ae_curves.png', dpi=110); plt.show()

## 12. Lưu kết quả và mô hình

In [ ]:
for (name, scene), (model, (mu, sd)) in AE_MODELS.items():
    if name != 'AE-flow·mask+density':
        continue
    torch.save(model.state_dict(), WORK / f'ae_umn_scene{scene + 1}.pth')
    np.savez(WORK / f'ae_umn_scene{scene + 1}_norm.npz', mean=mu, std=sd)
np.savez_compressed(WORK / 'umn_scores.npz', label=label, split=split.astype(str), scene=scene_id, clip=clip_id,
                    **{m.replace('·', '_').replace(' ', '_'): s for m, s in SMOOTHED.items()})
json.dump(dict(counter_checkpoint=str(WEIGHT_PATH), protocol='Marsden et al. ICIP 2016: 200 khung đầu mỗi clip để train', grid=GRID, win=WIN, nbins=NBINS,
               train_frames=TRAIN_FRAMES, val_frames=VAL_FRAMES, smooth=SMOOTH, topk=TOPK, infer_width=INFER_WIDTH,
               ae_epochs=AE_EPOCHS, ae_lr=AE_LR, seed=SEED,
               results=results.round(4).reset_index().to_dict(orient='records')),
          open(WORK / 'umn_config_results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(results.round(3).to_string())
print('\nFile đã lưu:', *sorted(p.name for p in WORK.iterdir() if p.suffix in ('.png', '.csv', '.json', '.pth', '.npz')), sep='\n  ')

## Cách đọc kết quả

- **Bảng ablation:**
  - AE-flow·mask so với AE-flow: cho biết lấy flow trong mask người có giúp phát hiện tốt hơn không.
  - AE-flow·mask+density so với AE-flow·mask: cho biết thêm kênh mật độ có giúp thêm không.
- **So với B0:** cho biết mô hình học được có tốt hơn luật đặt ngưỡng tay đang chạy trong demo không.
- **So với bài báo cùng giao thức:** Marsden và cộng sự (ICIP 2016) đạt AUC 0,929, với đặc trưng toàn khung (tốc độ, mật độ, độ đồng bộ) và GMM. Trên 0,929 là tốt hơn; khoảng 0,97–0,98 là ngang các số mà bài đó trích lại (SFM, Commotion Measure).
- **Giới hạn của UMN:** chỉ có một kiểu bất thường (chạy tán loạn) và người khá thưa. Muốn biết số báo nhầm mỗi giờ trên camera thật thì cần đo thêm trên video Dublin bình thường.